In [1]:
!pip install transformers x-transformers sentencepiece --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.0 MB/s eta 0:00:00


In [2]:
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install tokenizers==0.20.3 transformers==4.46.3

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 2.8 MB/s eta 0:00:000:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 91.2 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 80.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 48.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 98.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 15.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 34.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.

In [3]:
from sklearn.metrics import accuracy_score

In [4]:
import re
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from transformers import AutoTokenizer, AutoModel
from x_transformers import Decoder


In [5]:
BATCH_SIZE = 64
LR = 2e-4
EPOCHS = 100
MAX_LEN = 256

DECODER_DIM = 768
DECODER_DEPTH = 2
DECODER_HEADS = 8

labels = [
    "Happiness",
    "Sadness",
    "Anger",
    "Disgust",
    "Surprise",
    "Fear"
]

NUM_CLASSES = len(labels)

In [6]:
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

In [7]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
df = pd.read_csv("/kaggle/input/datasets/kmkimmy/sied-thai/SIED-Thai.csv")

df = df[['Tweets'] + labels].dropna()

# ❗ baseline: ไม่ลบ sample ว่าง (ให้ model เรียนเอง)
# df = df[df[labels].sum(axis=1) > 0]

In [9]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df['text'] = df['Tweets'].apply(clean_text)

In [10]:
train_df, test_df = train_test_split(df, test_size=0.2)
train_df, val_df  = train_test_split(train_df, test_size=0.1)

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


tokenizer_config.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/905k [00:00<?, ?B/s]

In [12]:
class EmotionDataset(Dataset):

    def __init__(self, df):
        self.texts = df['text'].tolist()
        self.labels = df[labels].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = tokenizer(
            self.texts[idx],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return (
            encoding["input_ids"].squeeze(0),
            encoding["attention_mask"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

train_loader = DataLoader(EmotionDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(EmotionDataset(val_df), batch_size=BATCH_SIZE)
test_loader  = DataLoader(EmotionDataset(test_df), batch_size=BATCH_SIZE)

In [13]:
class Model(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(MODEL_NAME)

        self.decoder = Decoder(
            dim=DECODER_DIM,
            depth=DECODER_DEPTH,
            heads=DECODER_HEADS
        )

        self.fc = nn.Linear(DECODER_DIM, NUM_CLASSES)

    def forward(self, input_ids, attention_mask):

        x = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state

        x = self.decoder(
            x,
            mask=attention_mask.bool()
        )

        # mean pooling
        mask = attention_mask.unsqueeze(-1).float()
        x = (x * mask).sum(1) / mask.sum(1)

        return self.fc(x)

model = Model().to(DEVICE)

#แยก LR: encoder ต่ำ, decoder/fc สูง
optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": 2e-5},
    {"params": model.decoder.parameters(), "lr": 1e-3},
    {"params": model.fc.parameters(), "lr": 1e-3},
], weight_decay=0.01)

criterion = nn.BCEWithLogitsLoss()

#Warm-up scheduler
from transformers import get_linear_schedule_with_warmup

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)


2026-05-15 07:53:25.587616: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778831605.770914      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778831605.823356      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778831606.265159      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778831606.265197      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778831606.265200      57 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/423M [00:00<?, ?B/s]

In [14]:
@torch.no_grad()
def evaluate(loader):
    
    model.eval()
    
    preds = []
    
    targets = []
    
    for input_ids, attention_mask, y in loader:
        
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        
        logits = model(input_ids, attention_mask)
        
        probs = torch.sigmoid(logits)
        
        # ❗ baseline threshold = 0.5
        pred = (probs > 0.5).int().cpu().numpy()
        preds.append(pred)
        
        targets.append(y.numpy())
        
    y_pred = np.vstack(preds)
    y_true = np.vstack(targets)
    
    f1_micro = f1_score(y_true, y_pred, average="micro")
    f1_macro = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    
    return f1_micro, f1_macro, acc

In [15]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for input_ids, attention_mask, y in train_loader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        y = y.float().to(DEVICE)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    f1_micro, f1_macro, acc = evaluate(val_loader)

    print(
        f"Epoch {epoch+1} | "
        f"loss={total_loss:.4f} | "
        f"f1_micro={f1_micro:.4f} | "
        f"f1_macro={f1_macro:.4f} | "
        f"acc={acc:.4f}"
    )


Epoch 1 | loss=11.1071 | f1_micro=0.6632 | f1_macro=0.5013 | acc=0.3750
Epoch 2 | loss=9.4939 | f1_micro=0.7276 | f1_macro=0.4804 | acc=0.5052
Epoch 3 | loss=8.0312 | f1_micro=0.7152 | f1_macro=0.4614 | acc=0.5000
Epoch 4 | loss=6.7198 | f1_micro=0.7322 | f1_macro=0.5359 | acc=0.4740
Epoch 5 | loss=5.9269 | f1_micro=0.7312 | f1_macro=0.5370 | acc=0.4948


In [16]:
f1_micro, f1_macro, acc = evaluate(test_loader)

print("\nFINAL TEST")
print("F1 Micro :", f1_micro)
print("F1 Macro :", f1_macro)
print("Accuracy :", acc)



FINAL TEST
F1 Micro : 0.7476489028213166
F1 Macro : 0.5322930547794273
Accuracy : 0.5020833333333333


In [17]:
#บันทึกโมเดล
torch.save(model.state_dict(), "wangchanberta_xtransformer_sied.pt")
print("Model saved to wangchanberta_xtransformer_sied.pt")


Model saved to wangchanberta_xtransformer_sied.pt


In [18]:
#โหลดโมเดลจาก checkpoint
loaded_model = Model().to(DEVICE)
loaded_model.load_state_dict(
    torch.load("wangchanberta_xtransformer_sied.pt", map_location=DEVICE)
)
loaded_model.eval()
print("Model loaded successfully!")


/tmp/ipykernel_57/227172610.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("wangchanberta_xtransformer_sied.pt", map_location=DEVICE)


Model loaded successfully!


In [19]:
from sklearn.metrics import classification_report, precision_score, recall_score

@torch.no_grad()
def predict_and_report(model, loader):
    model.eval()
    all_preds = []
    all_targets = []

    for input_ids, attention_mask, y in loader:
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits)
        pred = (probs > 0.5).int().cpu().numpy()

        all_preds.append(pred)
        all_targets.append(y.numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    #Per-label Precision / Recall / F1-score
    print("=" * 60)
    print("Per-Label Classification Report")
    print("=" * 60)
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=labels,
            digits=4,
            zero_division=0
        )
    )

    #Overall metrics
    f1_micro = f1_score(y_true, y_pred, average="micro")
    f1_macro = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    print("=" * 60)
    print(f"F1 Micro  : {f1_micro:.4f}")
    print(f"F1 Macro  : {f1_macro:.4f}")
    print(f"Accuracy  : {acc:.4f}")
    print("=" * 60)

# เรียกใช้กับ loaded model
predict_and_report(loaded_model, test_loader)


Per-Label Classification Report
              precision    recall  f1-score   support

   Happiness     0.6071    0.5484    0.5763        31
     Sadness     0.9032    0.9262    0.9146       393
       Anger     0.6667    0.4545    0.5405        66
     Disgust     0.5890    0.4175    0.4886       103
    Surprise     0.8750    0.1944    0.3182        36
        Fear     0.4444    0.2963    0.3556        54

   micro avg     0.8044    0.6984    0.7476       683
   macro avg     0.6809    0.4729    0.5323       683
weighted avg     0.7818    0.6984    0.7232       683
 samples avg     0.7806    0.7146    0.7206       683

F1 Micro  : 0.7476
F1 Macro  : 0.5323
Accuracy  : 0.5021
